# 01 — Coleta e integração dos dados

**Trabalho 2 – Previsão do PIB dos Municípios da Região Sul (PR, SC, RS)**

Este notebook:

1. baixa todas as fontes do IBGE pela API do SIDRA (código em `src/coleta.py`);
2. verifica a cobertura de cada fonte: todos os 1.191 municípios do Sul estão presentes em todos os anos?;
3. identifica valores ausentes, símbolos especiais e inconsistências nos dados brutos.

A integração em um painel município × ano (Fase 2) é feita na segunda parte deste notebook.

| Fonte | Tabela SIDRA | Nível | Uso no projeto |
|---|---|---|---|
| PIB dos Municípios | 5938 | Município | variável-alvo (PIB) e defasagens do PIB e do VAB por setor |
| Produção Agrícola Municipal (PAM) | 5457 | Município | valor e área das lavouras (total e principais culturas) |
| Pesquisa da Pecuária Municipal (PPM) | 3939 e 74 | Município | rebanhos e valor da produção de origem animal |
| Estimativas de População + Censos | 6579, 793, 202, 4709 | Município | população e PIB per capita |
| Pesquisa Industrial Anual – Empresa (PIA) | 5603 e 10455 | UF | contexto industrial do estado |
| Pesquisa Anual de Serviços (PAS) | 2715 e 10762 | UF | contexto de serviços do estado |

**Símbolos especiais do SIDRA:** `-` = zero absoluto, `..` = não se aplica, `...` = não disponível, `X` = omitido por sigilo.
O valor original fica preservado na coluna `valor_raw`. A função `para_numerico` converte `-` em 0 e os demais símbolos em `NaN`.

In [1]:
import sys
from pathlib import Path

# Permite importar o pacote src/ tanto rodando de notebooks/ quanto da raiz do projeto
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd

from src import coleta
from src.coleta import DIR_RAW, SIMBOLOS_SIDRA, para_numerico

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Coleta

`coleta_tudo()` baixa cada fonte para `data/raw/<fonte>.csv`. Arquivos que já existem são reaproveitados.
Para baixar tudo de novo, use `coleta_tudo(force=True)` ou, no terminal, `python -m src.coleta --force`.
O primeiro download leva alguns minutos, porque as tabelas municipais são baixadas ano a ano para respeitar o limite de valores por consulta da API.

In [2]:
caminhos = coleta.coleta_tudo()
pd.DataFrame(
    {"arquivo": [p.name for p in caminhos.values()],
     "tamanho_MB": [p.stat().st_size / 1e6 for p in caminhos.values()]},
    index=list(caminhos),
)

[ok] municipios_sul: já existe (municipios_sul.csv)
[ok] pib_municipios: já existe (pib_municipios.csv)
[ok] pam_lavouras: já existe (pam_lavouras.csv)
[ok] ppm_rebanhos: já existe (ppm_rebanhos.csv)
[ok] ppm_origem_animal: já existe (ppm_origem_animal.csv)
[ok] populacao: já existe (populacao.csv)
[ok] pia_empresa_uf: já existe (pia_empresa_uf.csv)
[ok] pas_uf: já existe (pas_uf.csv)


,arquivo,tamanho_MB
municipios_sul,municipios_sul.csv,0.13
pib_municipios,pib_municipios.csv,24.70
pam_lavouras,pam_lavouras.csv,117.67
ppm_rebanhos,ppm_rebanhos.csv,25.79
ppm_origem_animal,ppm_origem_animal.csv,36.74
populacao,populacao.csv,2.95
pia_empresa_uf,pia_empresa_uf.csv,0.03
pas_uf,pas_uf.csv,0.02


## 2. Funções auxiliares de verificação

In [3]:
def carrega(nome: str) -> pd.DataFrame:
    """Lê um CSV bruto e cria a coluna numérica `valor` a partir de `valor_raw`."""
    df = pd.read_csv(DIR_RAW / f"{nome}.csv", dtype={"valor_raw": str})
    if "valor_raw" in df.columns:
        df["valor"] = para_numerico(df["valor_raw"])
    return df


def conta_simbolos(df: pd.DataFrame) -> pd.Series:
    """Quantas vezes cada símbolo especial do SIDRA aparece."""
    s = df["valor_raw"].astype(str).str.strip()
    return s[s.isin(list(SIMBOLOS_SIDRA))].value_counts()


def cobertura_municipios(df: pd.DataFrame, cod_municipios: set) -> pd.DataFrame:
    """Por ano: municípios presentes, municípios com valor numérico e ausentes em relação à lista oficial."""
    linhas = []
    for ano, g in df.groupby("ano"):
        presentes = set(g["cod_local"])
        com_valor = set(g.loc[g["valor"].notna(), "cod_local"])
        linhas.append({
            "ano": ano,
            "presentes": len(presentes),
            "com_algum_valor": len(com_valor),
            "faltando_na_fonte": len(cod_municipios - presentes),
            "fora_da_lista_oficial": len(presentes - cod_municipios),
        })
    return pd.DataFrame(linhas).set_index("ano")

## 3. Municípios da Região Sul

Lista oficial da API de Localidades do IBGE. O **código IBGE de 7 dígitos** (`cod_ibge`) é a chave de integração entre todas as fontes.

In [4]:
mun = pd.read_csv(DIR_RAW / "municipios_sul.csv")
cod_sul = set(mun["cod_ibge"])

print(f"Total de municípios: {len(mun)} | códigos únicos: {mun['cod_ibge'].nunique()}")
display(mun["uf"].value_counts().rename("municípios").to_frame())
display(mun[["mesorregiao", "microrregiao", "regiao_intermediaria", "regiao_imediata"]].nunique()
        .rename("categorias distintas").to_frame())
mun.head()

Total de municípios: 1191 | códigos únicos: 1191


,municípios
uf,
RS,497
PR,399
SC,295


,categorias distintas
mesorregiao,23
microrregiao,94
regiao_intermediaria,21
regiao_imediata,96


,cod_ibge,municipio,cod_uf,uf,cod_mesorregiao,mesorregiao,cod_microrregiao,microrregiao,cod_regiao_intermediaria,regiao_intermediaria,cod_regiao_imediata,regiao_imediata
0,4100103,Abatiá,41,PR,4104,Norte Pioneiro Paranaense,41015,Cornélio Procópio,4105,Londrina,410022,Santo Antônio da Platina
1,4100202,Adrianópolis,41,PR,4110,Metropolitana de Curitiba,41035,Cerro Azul,4101,Curitiba,410001,Curitiba
2,4100301,Agudos do Sul,41,PR,4110,Metropolitana de Curitiba,41039,Rio Negro,4101,Curitiba,410001,Curitiba
3,4100400,Almirante Tamandaré,41,PR,4110,Metropolitana de Curitiba,41037,Curitiba,4101,Curitiba,410001,Curitiba
4,4100459,Altamira do Paraná,41,PR,4102,Centro Ocidental Paranaense,41004,Goioerê,4104,Maringá,410015,Campo Mourão


## 4. PIB dos Municípios (tabela 5938)

Valores a preços correntes, em **mil reais**. Variáveis: PIB, impostos líquidos de subsídios, VAB total e VAB por setor.

In [5]:
pib = carrega("pib_municipios")
print(pib.shape, "| anos:", pib["ano"].min(), "-", pib["ano"].max())
print("Símbolos especiais:", conta_simbolos(pib).to_dict())

so_pib = pib[pib["cod_variavel"] == 37]
cob_pib = cobertura_municipios(so_pib, cod_sul)
cob_pib.T

(183414, 9) | anos: 2002 - 2023
Símbolos especiais: {'...': 14523}


ano,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
presentes,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191
com_algum_valor,1188,1188,1188,1188,1188,1188,1188,1188,1188,1188,1188,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191
faltando_na_fonte,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
fora_da_lista_oficial,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [6]:
# Quais municípios não têm PIB em algum ano?
sem_pib = so_pib[so_pib["valor"].isna()]
sem_pib.groupby(["cod_local", "nome_local"])["ano"].agg(["min", "max", "count"])

,,min,max,count
cod_local,nome_local,,,
4212650,Pescaria Brava - SC,2002,2012,11
4220000,Balneário Rincão - SC,2002,2012,11
4314548,Pinto Bandeira - RS,2002,2012,11


### 4.1 Disponibilidade de cada variável por ano

Contamos quantos municípios têm valor numérico em cada variável e ano.

In [7]:
disp = (pib.assign(tem_valor=pib["valor"].notna())
           .pivot_table(index="variavel", columns="ano", values="tem_valor", aggfunc="sum"))
disp.loc[:, 2018:]

ano,2018,2019,2020,2021,2022,2023
variavel,,,,,,
"Impostos, líquidos de subsídios, sobre produtos a preços correntes",1191,1191,1191,1191,0,0
Produto Interno Bruto a preços correntes,1191,1191,1191,1191,1191,1191
"Valor adicionado bruto a preços correntes da administração, defesa, educação e saúde públicas e seguridade social",1191,1191,1191,1191,0,0
Valor adicionado bruto a preços correntes da agropecuária,1191,1191,1191,1191,0,0
Valor adicionado bruto a preços correntes da indústria,1191,1191,1191,1191,0,0
"Valor adicionado bruto a preços correntes dos serviços, exclusive administração, defesa, educação e saúde públicas e seguridade social",1191,1191,1191,1191,0,0
Valor adicionado bruto a preços correntes total,1191,1191,1191,1191,0,0


> ⚠️ **Limitação importante:** para **2022 e 2023 o IBGE só publicou o PIB total** (o VAB por setor e os impostos aparecem como `...` para todos os municípios, e até no nível de UF).
> Conferimos também a base completa do FTP do IBGE (`Pib_Municipios/2022_2023/base/base_de_dados_2010_2023_txt.zip`), e nela os campos de VAB e impostos de 2022–2023 também estão em branco.
>
> **Consequência para a modelagem:** a estrutura setorial do município (participação de agro, indústria, serviços e administração pública) só está disponível **até 2021**.
> Para prever 2024, a informação setorial mais recente tem 3 anos de defasagem. Para que treino e previsão sejam coerentes, as variáveis setoriais entram com **defasagem de 3 anos** (t-3) em todos os anos de treino.

### 4.2 Consistência contábil

Pela definição do IBGE, **PIB = VAB total + impostos líquidos** e **VAB total = soma dos 4 setores**.
Conferimos as duas identidades nos anos em que há abertura setorial (2002–2021). Pequenas diferenças são esperadas por causa do arredondamento em mil reais.

In [8]:
nomes_pib = {37: "pib", 543: "impostos", 498: "vab_total", 513: "vab_agro",
             517: "vab_industria", 6575: "vab_servicos", 525: "vab_adm_publica"}
largo = (pib.assign(var=pib["cod_variavel"].map(nomes_pib))
            .pivot_table(index=["cod_local", "ano"], columns="var", values="valor", aggfunc="first"))

dif_pib = largo["pib"] - (largo["vab_total"] + largo["impostos"])
dif_vab = largo["vab_total"] - largo[["vab_agro", "vab_industria", "vab_servicos", "vab_adm_publica"]].sum(axis=1)

pd.DataFrame({
    "PIB - (VAB + impostos)": dif_pib.abs().describe(),
    "VAB - soma dos setores": dif_vab.abs().describe(),
})

,PIB - (VAB + impostos),VAB - soma dos setores
count,"23,787.00","23,787.00"
mean,0.25,0.41
std,0.43,0.50
min,0.00,0.00
25%,0.00,0.00
50%,0.00,0.00
75%,0.00,1.00
max,1.00,2.00


In [9]:
# Valores negativos (possíveis em impostos líquidos de subsídios ou em VAB de algum setor)
negativos = (largo < 0).sum()
negativos[negativos > 0].rename("observações negativas").to_frame()

,observações negativas
var,
vab_industria,4


In [10]:
# Evolução do PIB nominal total da região, em bilhões de reais
pib_total = largo.groupby("ano")["pib"].sum() / 1e6
pib_total.to_frame("PIB Sul (R$ bi)").assign(cresc_pct=lambda d: d["PIB Sul (R$ bi)"].pct_change() * 100).T

ano,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
PIB Sul (R$ bi),241.56,293.46,328.26,345.38,376.33,436.95,497.39,530.12,620.18,696.25,765.00,880.29,948.45,"1,008.04","1,067.36","1,122.04","1,195.55","1,272.11","1,308.15","1,559.83","1,674.52","1,834.42"
cresc_pct,NaN,21.48,11.86,5.21,8.96,16.11,13.83,6.58,16.99,12.27,9.88,15.07,7.74,6.28,5.89,5.12,6.55,6.40,2.83,19.24,7.35,9.55


## 5. Produção Agrícola Municipal (tabela 5457)

Lavouras temporárias e permanentes: total e as principais culturas do Sul.
Valor da produção em mil reais (a partir de 1994).

In [11]:
pam = carrega("pam_lavouras")
col_prod = "Produto das lavouras temporárias e permanentes"
print(pam.shape, "| anos:", pam["ano"].min(), "-", pam["ano"].max())
print("Símbolos especiais:", conta_simbolos(pam).to_dict())

valor_total = pam[(pam["cod_variavel"] == 215) & (pam[col_prod] == "Total")]
cobertura_municipios(valor_total, cod_sul).T

(1205292, 11) | anos: 2002 - 2024
Símbolos especiais: {'-': 362266, '..': 27393, '...': 4300}


ano,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
presentes,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191
com_algum_valor,1187,1187,1187,1187,1187,1188,1186,1185,1186,1187,1185,1188,1187,1187,1187,1187,1187,1187,1185,1186,1186,1187,1187
faltando_na_fonte,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
fora_da_lista_oficial,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [12]:
# Valor da produção (R$ bi) por cultura e ano: as quebras de safra aparecem aqui
valor_cultura = (pam[pam["cod_variavel"] == 215]
                 .pivot_table(index=col_prod, columns="ano", values="valor", aggfunc="sum") / 1e6)
valor_cultura.loc[:, 2015:].sort_values(2024, ascending=False).round(1)

ano,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
Produto das lavouras temporárias e permanentes,,,,,,,,,,
Total,75.10,89.40,86.10,90.10,91.80,110.30,191.30,168.20,175.60,167.30
Soja (em grão),34.00,40.50,40.50,45.70,42.30,50.60,108.20,74.70,87.00,76.00
Milho (em grão),8.90,12.30,9.80,10.00,12.80,17.60,23.60,28.40,23.30,19.50
Arroz (em casca),7.00,7.00,8.00,7.00,7.00,9.20,16.20,12.90,14.30,17.50
Fumo (em folha),5.80,5.70,6.80,6.30,6.50,5.80,6.60,8.20,10.40,11.90
Trigo (em grão),2.70,3.50,1.90,3.30,3.40,5.60,9.80,13.90,6.10,7.40
Mandioca,2.10,2.70,3.20,3.00,2.50,2.60,3.00,3.80,4.90,4.10
Feijão (em grão),1.90,2.40,2.20,1.60,2.00,2.60,3.40,3.70,3.40,3.90
Maçã,1.30,1.60,1.60,1.40,1.80,1.70,2.30,1.90,2.90,3.10


## 6. Pesquisa da Pecuária Municipal (tabelas 3939 e 74)

In [13]:
reb = carrega("ppm_rebanhos")
col_reb = "Tipo de rebanho"
print("Rebanhos:", reb.shape, "| símbolos:", conta_simbolos(reb).to_dict())
display(cobertura_municipios(reb, cod_sul).T)

(reb.pivot_table(index=col_reb, columns="ano", values="valor", aggfunc="sum") / 1e6).loc[:, 2018:].round(2)

Rebanhos: (273930, 11) | símbolos: {'-': 45155, '...': 360}


ano,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
presentes,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191
com_algum_valor,1188,1188,1186,1188,1188,1188,1188,1188,1188,1188,1188,1190,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191
faltando_na_fonte,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
fora_da_lista_oficial,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


ano,2018,2019,2020,2021,2022,2023,2024
Tipo de rebanho,,,,,,,
Bovino,26.12,25.39,24.12,23.68,24.37,25.33,24.55
Bubalino,0.10,0.10,0.09,0.09,0.10,0.10,0.10
Caprino,0.22,0.21,0.19,0.18,0.17,0.16,0.14
Codornas,2.90,2.85,2.44,2.21,2.70,3.11,3.00
Equino,0.92,0.94,0.91,0.88,0.84,0.83,0.82
Galináceos - galinhas,61.47,60.49,62.49,63.81,64.76,65.04,68.23
Galináceos - total,688.77,668.18,696.67,739.93,758.71,754.93,747.35
Ovino,4.01,3.96,3.86,3.94,4.25,4.22,3.82
Suíno - matrizes de suínos,2.02,2.00,2.11,2.19,2.15,2.17,2.25


In [14]:
ani = carrega("ppm_origem_animal")
col_ani = "Tipo de produto de origem animal"
print("Origem animal:", ani.shape, "| símbolos:", conta_simbolos(ani).to_dict())
valor_ani = ani[ani["cod_variavel"] == 215]
display(cobertura_municipios(valor_ani[valor_ani[col_ani] == "Total"], cod_sul).T)

# Valor da produção de origem animal (R$ bi)
(valor_ani.pivot_table(index=col_ani, columns="ano", values="valor", aggfunc="sum") / 1e6).loc[:, 2018:].round(2)

Origem animal: (383502, 11) | símbolos: {'-': 97396, '..': 27393, '...': 468}


ano,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
presentes,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191
com_algum_valor,1188,1188,1186,1188,1188,1188,1188,1188,1188,1188,1188,1190,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191,1191
faltando_na_fonte,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
fora_da_lista_oficial,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


ano,2018,2019,2020,2021,2022,2023,2024
Tipo de produto de origem animal,,,,,,,
Casulos do bicho-da-seda,0.05,0.05,0.05,0.04,0.04,0.04,0.04
Leite,13.74,14.91,19.76,23.44,27.91,27.86,30.38
Lã,0.09,0.07,0.07,0.08,0.08,0.05,0.05
Mel de abelha,0.22,0.22,0.26,0.32,0.35,0.33,0.36
Ovos de codorna,0.06,0.05,0.06,0.06,0.08,0.10,0.10
Ovos de galinha,3.08,3.13,3.87,4.32,5.27,6.21,6.78
Total,17.23,18.43,24.07,28.26,33.73,34.61,37.72


## 7. População (tabelas 6579, 793, 202 e 4709)

As estimativas anuais não existem nos anos de censo e contagem. Esses anos vêm da fonte oficial correspondente.
**O IBGE não publicou estimativa para 2023**, então esse ano será interpolado no pré-processamento.

In [15]:
pop = carrega("populacao")
print("Duplicados (município, ano):", pop.duplicated(["cod_local", "ano"]).sum())

resumo_pop = pop.groupby(["ano", "fonte"]).agg(
    municipios=("cod_local", "nunique"),
    sem_valor=("valor", lambda s: s.isna().sum()),
    pop_total_milhoes=("valor", lambda s: s.sum() / 1e6),
)
anos_faltando = sorted(set(range(coleta.ANO_INICIAL, coleta.ANO_FINAL + 1)) - set(pop["ano"]))
print("Anos sem nenhuma fonte de população:", anos_faltando)
resumo_pop

Duplicados (município, ano): 0
Anos sem nenhuma fonte de população: [2023]


,,municipios,sem_valor,pop_total_milhoes
ano,fonte,,,
2002,Estimativas de População (6579),1191,3,25.73
2003,Estimativas de População (6579),1191,3,26.03
2004,Estimativas de População (6579),1191,3,26.64
2005,Estimativas de População (6579),1191,3,26.97
2006,Estimativas de População (6579),1191,3,27.31
2007,Contagem da População 2007 (793),1188,0,26.73
2008,Estimativas de População (6579),1191,3,27.50
2009,Estimativas de População (6579),1191,3,27.72
2010,Censo 2010 (202),1188,0,27.39


In [16]:
# Municípios sem população em algum ano
pop[pop["valor"].isna()].groupby(["cod_local", "nome_local"])["ano"].agg(["min", "max", "count"])

,,min,max,count
cod_local,nome_local,,,
4212650,Pescaria Brava - SC,2002,2011,8
4220000,Balneário Rincão - SC,2002,2011,8
4314548,Pinto Bandeira - RS,2002,2011,8


## 8. PIA-Empresa e PAS por UF: quebra de série em 2024

As duas pesquisas só existem por UF. Em 2024 o IBGE passou a publicar uma **série nova** (tabelas 10455 e 10762).
Comparamos 2022 e 2023 (série antiga) com 2024 (série nova) para ver se a variação é plausível ou se há um salto metodológico.

In [17]:
pia = carrega("pia_empresa_uf")
pia_largo = pia.pivot_table(index=["variavel", "nome_local"], columns="ano", values="valor", aggfunc="first")
pia_cmp = pia_largo[[2021, 2022, 2023, 2024]].copy()
pia_cmp["var_22_23_%"] = (pia_cmp[2023] / pia_cmp[2022] - 1) * 100
pia_cmp["var_23_24_%"] = (pia_cmp[2024] / pia_cmp[2023] - 1) * 100
pia_cmp

ano                                                                            2021           2022           2023           2024  var_22_23_%  var_23_24_%
variavel                                           nome_local                                                                                             
Número de unidades locais                          Paraná                 18,304.00      18,590.00      19,783.00            NaN         6.42          NaN
                                                   Rio Grande do Sul      18,882.00      18,927.00      19,445.00            NaN         2.74          NaN
                                                   Santa Catarina         20,316.00      20,194.00      21,145.00            NaN         4.71          NaN
Número de unidades locais de empresas com 5 ou ... Paraná                       NaN            NaN            NaN      19,850.00          NaN          NaN
                                                   Rio Grande do Sul            NaN            NaN            NaN      19,587.00          NaN          NaN
                                                   Santa Catarina               NaN            NaN            NaN      21,294.00          NaN          NaN
Pessoal ocupado em 31/12                           Paraná                682,815.00     694,644.00     715,482.00     730,562.00         3.00         2.11
                                                   Rio Grande do Sul     668,332.00     691,118.00     684,246.00     683,396.00        -0.99        -0.12
                                                   Santa Catarina        737,811.00     739,079.00     749,939.00     765,522.00         1.47         2.08
Receita líquida de vendas                          Paraná            439,941,320.00 535,842,455.00 516,848,190.00 555,756,745.00        -3.54         7.53
                                                   Rio Grande do Sul 403,772,095.00 483,531,905.00 454,982,264.00 463,598,368.00        -5.90         1.89
                                                   Santa Catarina    301,197,871.00 341,961,191.00 341,671,718.00 373,844,376.00        -0.08         9.42
Salários, retiradas e outras remunerações          Paraná             26,326,605.00  29,668,899.00  32,670,364.00  35,148,947.00        10.12         7.59
                                                   Rio Grande do Sul  27,152,427.00  31,330,004.00  34,046,264.00  35,858,005.00         8.67         5.32
                                                   Santa Catarina     27,651,975.00  31,664,799.00  34,293,319.00  37,296,337.00         8.30         8.76
Valor bruto da produção industrial                 Paraná            402,292,585.00 479,406,183.00 455,753,310.00 498,375,938.00        -4.93         9.35
                                                   Rio Grande do Sul 396,087,512.00 467,716,895.00 432,327,780.00 441,123,324.00        -7.57         2.03
                                                   Santa Catarina    300,731,416.00 334,648,778.00 334,880,877.00 369,013,392.00         0.07        10.19
Valor da transformação industrial                  Paraná            140,049,151.00 166,109,868.00 163,193,252.00 183,826,919.00        -1.76        12.64
                                                   Rio Grande do Sul 136,075,456.00 157,037,533.00 150,465,499.00 159,547,288.00        -4.19         6.04
                                                   Santa Catarina    113,343,287.00 121,812,501.00 127,911,440.00 142,462,573.00         5.01        11.38

In [18]:
pas = carrega("pas_uf")
pas_largo = pas.pivot_table(index=["variavel", "cod_local"], columns="ano", values="valor", aggfunc="first")
pas_cmp = pas_largo[[2021, 2022, 2023, 2024]].copy()
pas_cmp["var_22_23_%"] = (pas_cmp[2023] / pas_cmp[2022] - 1) * 100
pas_cmp["var_23_24_%"] = (pas_cmp[2024] / pas_cmp[2023] - 1) * 100
pas_cmp

ano                                                                    2021           2022           2023           2024  var_22_23_%  var_23_24_%
variavel                                           cod_local                                                                                      
Gastos com pessoal - salários, retiradas e outr... 41                   NaN            NaN            NaN  37,604,491.00          NaN          NaN
                                                   42                   NaN            NaN            NaN  30,380,829.00          NaN          NaN
                                                   43                   NaN            NaN            NaN  31,781,726.00          NaN          NaN
Número de empresas                                 41            129,969.00     131,178.00     152,473.00     153,024.00        16.23         0.36
                                                   42            108,868.00     119,856.00     127,078.00     129,391.00         6.03         1.82
                                                   43            112,911.00     120,129.00     101,198.00     135,694.00       -15.76        34.09
Pessoal ocupado em 31/12                           41            875,874.00     948,791.00   1,010,743.00   1,031,479.00         6.53         2.05
                                                   42            709,228.00     716,321.00     813,367.00     801,391.00        13.55        -1.47
                                                   43            751,049.00     785,644.00     796,056.00     838,785.00         1.33         5.37
Receita bruta de serviços                          41        124,881,711.00 157,952,463.00 189,942,725.00 218,663,487.00        20.25        15.12
                                                   42        108,219,771.00 138,814,948.00 158,378,591.00 189,234,509.00        14.09        19.48
                                                   43        114,192,540.00 142,960,460.00 163,094,275.00 185,983,672.00        14.08        14.03
Salários, retiradas e outras remunerações          41         24,882,592.00  30,020,634.00  34,171,919.00            NaN        13.83          NaN
                                                   42         20,254,405.00  24,576,276.00  28,258,603.00            NaN        14.98          NaN
                                                   43         22,260,864.00  26,218,766.00  29,295,161.00            NaN        11.73          NaN

## 9. Conclusões parciais da coleta

**Cobertura**
- As 8 fontes foram baixadas pela API, e **todos os 1.191 municípios do Sul aparecem em todas as fontes municipais e em todos os anos**. Nenhum código fica fora da lista oficial.
- Os únicos buracos estruturais são **Pescaria Brava, Balneário Rincão (SC) e Pinto Bandeira (RS)**, instalados em 2013. Eles não têm PIB antes de 2013 nem população antes de 2012. Como a modelagem usa anos recentes, isso não causa perda: a partir de 2013 os 1.191 municípios têm PIB.

**PIB (alvo)**
- O PIB total existe de 2002 a 2023 e está completo a partir de 2013.
- **O VAB por setor e os impostos só existem até 2021.** Em 2022 e 2023 o IBGE publicou apenas o PIB total, como confirmado no SIDRA e na base do FTP. As variáveis setoriais vão entrar com defasagem de 3 anos.
- As identidades contábeis fecham com diferença máxima de 1–2 mil reais, só arredondamento. Há 4 observações de VAB da indústria negativo, a examinar na EDA.
- O PIB nominal da região cresceu **7,4% em 2022 e 9,6% em 2023**. Com crescimento nominal desse tamanho, a maior parte dos municípios deve cair em "Forte crescimento" (> 5%), e as classes ficam muito desbalanceadas.

**Agropecuária**
- A PAM e a PPM vão até 2024, ou seja, existem para o ano a ser previsto. O valor da produção agrícola mostra choques fortes: o recorde de 2021 (R$ 191 bi), a queda com a estiagem de 2022 e a nova queda em 2024 (R$ 167 bi, com a soja caindo de 87 para 76 bi).
- Na PAM, o símbolo `-` (zero absoluto) é muito comum, porque o município não produz aquela cultura, e é convertido em 0. `..` e `...` viram `NaN`.

**População**
- 2023 não tem estimativa oficial e será interpolado entre o Censo 2022 e a estimativa de 2024.
- **Quebra de série:** a estimativa de 2021 (30,4 milhões) é maior que o Censo 2022 (29,9 milhões), e o mesmo acontece entre 2006 e a Contagem 2007. As estimativas anteriores a um censo superestimavam a população, então a variação populacional nesses anos não é crescimento real e precisa ser tratada (ex.: não usar a variação 2021→2022 como variável).

**PIA e PAS (UF)**
- A série nova de 2024 tem variações plausíveis em relação a 2023 (ex.: receita de serviços +14% a +19%, receita industrial +2% a +9%), então dá para usar como contexto estadual.
- Os nomes e códigos de algumas variáveis mudam entre as séries (ex.: unidades locais, salários 673 → 857) e precisam ser harmonizados.
- O número de empresas de serviços do RS oscila de forma estranha (-16% em 2023, +34% em 2024). Variáveis de receita são mais confiáveis que contagens.